# Coding practice: which tokens instruction tuning supervises

Instruction tuning trains on conversations, but not on all of a conversation. The system and user turns stay in the context that predicts the reply, and the loss falls on the assistant's tokens only.

Here you build the mask that picks out those positions, and report the loss under it.

<details style="border:1px solid #e5e7eb;border-radius:8px;padding:10px 14px;background:#f9fafb;color:#111827;margin:14px 0;">
<summary style="cursor:pointer;font-weight:600;">Hint: which position predicts which token</summary>

Position `i` predicts the token at position `i + 1`, so there are `len(roles) - 1` targets. Ask whose token each target is.

</details>

In [ ]:
# One tokenized conversation over two turns. Each entry is (role, per-token NLL
# under the current model). Roles are per token, not per turn, because the mask
# is built position by position.
conversation = [
    ("system", 0.9), ("system", 0.7),
    ("user", 1.7), ("user", 1.4), ("user", 1.5),
    ("assistant", 0.9), ("assistant", 0.7),
    ("user", 1.6), ("user", 1.3),
    ("assistant", 0.8), ("assistant", 0.6),
]

roles = [role for role, _ in conversation]
losses = [loss for _, loss in conversation]
print("tokens:", len(roles), "| assistant tokens:", roles.count("assistant"))

In [ ]:
def target_mask(roles, supervised_role="assistant"):
    """Return the 0/1 mask over TARGETS for a next-token objective."""
    # TODO: return a list of 0/1 over the targets, with 1 where the target is supervised.
    raise NotImplementedError("Build the target mask.")


def supervised_mean_nll(losses, mask):
    """Mean NLL over supervised targets only, excluding the rest from sum and count."""
    # TODO: average the losses of the targets the mask selects.
    raise NotImplementedError("Average over the supervised targets.")


targets = losses[1:]
mask = target_mask(roles)

assert len(mask) == len(roles) - 1, "one mask entry per target, not per token"
assert sum(mask) == 4, "there are four assistant tokens for the model to predict"
# Two toy checks on the averaging, independent of the mask.
assert abs(supervised_mean_nll([2.0, 4.0], [1, 1]) - 3.0) < 1e-12
assert abs(supervised_mean_nll([2.0, 4.0], [0, 1]) - 4.0) < 1e-12
print("supervised mean NLL =", round(supervised_mean_nll(targets, mask), 4))

In [ ]:
# Three numbers a training loop might report for the same conversation.
print("mean over every target       =", round(sum(targets) / len(targets), 4))
print("sum over supervised targets  =", round(sum(t for t, m in zip(targets, mask) if m), 4))
print("mean over supervised targets =", round(supervised_mean_nll(targets, mask), 4))

# Now the off-by-one: a mask read from each token's own role rather than from
# the role of the token it predicts.
unshifted = [1 if role == "assistant" else 0 for role in roles][:-1]
print("\nshifted mask  :", mask)
print("unshifted mask:", unshifted)
print("unshifted mean =", round(supervised_mean_nll(targets, unshifted), 4))

included = [i for i in range(len(targets)) if unshifted[i] and not mask[i]]
dropped = [i for i in range(len(targets)) if mask[i] and not unshifted[i]]
print("targets it adds :", [(i, targets[i], roles[i + 1]) for i in included])
print("targets it loses:", [(i, targets[i], roles[i + 1]) for i in dropped])

## Interpret the outputs

- The cell above printed three numbers for the same conversation. Which would you report as the training loss, and what would each of the other two tell you instead?
- The mean over every target averages in positions the model is never asked to produce. What does that number tell you about the replies?
- Read the lists of targets the unshifted mask adds and loses. Which turn boundary does each sit on? Would this bug show up as a training loss that stops falling?
- Which of the sum and the mean would you compare across conversations of different lengths, and why?